In [7]:
from flask import Flask, jsonify
import sqlite3

app = Flask(__name__)

# SQLite 연결 설정
def get_db_connection():
    conn = sqlite3.connect('/mnt/data/cruise_schedule.db')
    return conn

# API 엔드포인트: 모든 예약 정보를 JSON 형식으로 반환
@app.route('/api/reservations', methods=['GET'])
def get_reservations():
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM cruise_schedule")
    rows = cursor.fetchall()
    columns = ['zone', 'site_name', 'ship_name', 'date', 'wave_power', 'fish_name', 'reservation', 'booking_url']
    results = [dict(zip(columns, row)) for row in rows]
    conn.close()
    return jsonify(results)

if __name__ == "__main__":
    app.run(host='127.0.0.1', port=5000)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit


In [14]:
from flask import Flask, render_template, jsonify, request
import sqlite3
import datetime
from flask_cors import CORS

app = Flask(__name__)
CORS(app)
# 루트 경로: 홈페이지 렌더링
@app.route('/')
def home():
    return render_template('boot.html')  # templates/boot.html 위치에 있어야 함

# 예약 API: /api/reservations?date=YYYY-MM-DD
@app.route('/api/reservations', methods=['GET'])
def get_reservations():
    date_param = request.args.get('date')  # URL 쿼리에서 날짜 가져오기
    print(f"📅 요청 날짜: {date_param} ({type(date_param)})")  # 콘솔 출력 (디버깅용)

    conn = sqlite3.connect('cruise_schedule.db')
    cursor = conn.cursor()

    # 날짜가 전달된 경우 필터링
    if date_param:
        try:
            parts = date_param.split("-")
            print(f"✅ parts: {parts}")  # 여기까지 안 나오면 split에서 문제
            year, month, day = int(parts[0]), int(parts[1]), int(parts[2])
            weekday_ko = ["월", "화", "수", "목", "금", "토", "일"]
            weekday = weekday_ko[datetime.date(year, month, day).weekday()]
            db_date = f"{year}년 {month}월{day}일({weekday})"
            print(f"db날짜형식:{db_date}")
            cursor.execute("SELECT * FROM cruise_schedule WHERE date = ?", (db_date,))
        except Exception as e:
            print(f"❌ 날짜 파싱 오류: {e}")
            return jsonify({"error": "날짜 형식이 잘못되었습니다."}), 400
    else:
        cursor.execute("SELECT * FROM cruise_schedule")

    rows = cursor.fetchall()
    columns = ['zone', 'site_name', 'ship_name', 'date', 'wave_power', 'fish_name', 'reservation', 'booking_url']
    results = [dict(zip(columns, row)) for row in rows]

    conn.close()
    return jsonify(results)

# favicon 요청 무시
@app.route('/favicon.ico')
def favicon():
    return '', 204

if __name__ == "__main__":
    app.run(host='127.0.0.1', port=5000, debug=True, use_reloader=True)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1

C:\Users\kang\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
